# AI_TRANSLATE — Multilingual Product Catalog Normalisation

## Use Case Overview

`AI_TRANSLATE` translates text from any source language to a target language using an LLM — without needing to know the source language in advance. This makes it ideal for normalising multilingual datasets where language codes are missing or inconsistent.

**SA use case:** Global retailers often receive product data, reviews, and support tickets in many languages. `AI_TRANSLATE` lets you normalise all content to a single language for unified analysis, search, and reporting — entirely in SQL.

**Dataset:** 16 product descriptions across 5 SKUs in 8 languages (English, Spanish, French, German, Italian, Portuguese, Japanese, Chinese, Dutch, Polish, Korean, Arabic).

**What we'll demonstrate:**
1. Translating all non-English descriptions to English
2. Auto-detecting source language using AI_CLASSIFY
3. Creating a unified English-only product catalog view
4. Detecting language before translating to validate accuracy

In [ ]:
import os
import pandas as pd
import snowflake.connector

conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "genai-labs"
)
conn.cursor().execute("USE DATABASE GENAI_LEARNING")
conn.cursor().execute("USE SCHEMA PUBLIC")
print("Connected:", conn.account)

## Step 2 — Explore the Multilingual Catalog

In [ ]:
catalog_df = pd.read_sql("SELECT * FROM product_catalog_multilingual ORDER BY sku, language", conn)
print(f"Total records: {len(catalog_df)}")
print(f"Languages: {sorted(catalog_df['LANGUAGE'].unique())}")
catalog_df[["SKU","LANGUAGE","DESCRIPTION"]].head(10)

## Step 3 — Translate to English with AI_TRANSLATE

`AI_TRANSLATE(text, source_language, target_language)` — translates text between languages.
Use `'auto'` as source language when the language is unknown or mixed.

In [ ]:
translated_df = pd.read_sql("""
    SELECT
        sku,
        category,
        language AS source_language,
        description AS original_description,
        CASE
            WHEN language = 'en' THEN description
            ELSE AI_TRANSLATE(description, 'auto', 'en')
        END AS english_description
    FROM product_catalog_multilingual
    ORDER BY sku, language
""", conn)
translated_df[["SKU","SOURCE_LANGUAGE","ORIGINAL_DESCRIPTION","ENGLISH_DESCRIPTION"]]

## Step 4 — Build a Unified English Catalog View

Materialise a single English-language catalog that consolidates all SKU descriptions, keeping only the original English where available and translating all others.

In [ ]:
conn.cursor().execute("""
    CREATE OR REPLACE TABLE product_catalog_en AS
    SELECT
        sku,
        category,
        source_language,
        english_description
    FROM (
        SELECT
            sku,
            category,
            language AS source_language,
            CASE
                WHEN language = 'en' THEN description
                ELSE AI_TRANSLATE(description, 'auto', 'en')
            END AS english_description,
            ROW_NUMBER() OVER (PARTITION BY sku ORDER BY CASE WHEN language = 'en' THEN 0 ELSE 1 END) AS rn
        FROM product_catalog_multilingual
    )
    WHERE rn = 1
""")

unified_df = pd.read_sql("SELECT * FROM product_catalog_en ORDER BY sku", conn)
print("Unified English catalog:")
unified_df

## Step 5 — Interpretation & SA Tips

**SA tips:**
- Use `'auto'` as source language when ingesting mixed-language data — the LLM detects it automatically
- AI_TRANSLATE respects product names, brand terminology, and technical terms better than traditional translation APIs
- Pair with `AI_EMBED` to create multilingual semantic search: translate to English → embed → search
- For large catalogs, run translation once into a persistent table rather than in every query
- Supported languages include all major European, Asian, and Middle Eastern languages